In [ ]:
# ============================================================
# Notebook Initialization
# Project: ECCO Procurement Data Quality Assessment
# ============================================================

# 1. Standard Library
import os
import sys
import json
import logging
import warnings
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter

# 2. Data Analysis
import pandas as pd
import numpy as np

# 3. Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# 4. PostgreSQL and SQLAlchemy
import psycopg2
from sqlalchemy import create_engine, inspect, text
from sqlalchemy.engine import URL
from sqlalchemy.exc import SQLAlchemyError

# 5. Environment Configuration
from dotenv import load_dotenv

# 6. Notebook Display
from IPython.display import display, Markdown

# ============================================================
# Display Configuration
# ============================================================

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 160)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["figure.dpi"] = 110

warnings.filterwarnings("default")

print("Notebook initialization completed.")
print(f"Python version: {sys.version.split()[0]}")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")


Notebook initialization completed.
Python version: 3.14.4
Pandas version: 3.1.0
NumPy version: 2.5.4


In [11]:

# ============================================================
# Database Configuration and Connection
# ============================================================

PROJECT_ROOT = Path.cwd()
ENV_FILE = PROJECT_ROOT / ".env"

load_dotenv(dotenv_path=ENV_FILE)

DB_HOST = os.getenv("PGHOST", "localhost")
DB_PORT = os.getenv("PGPORT", "5432")
DB_NAME = os.getenv("PGDATABASE")
DB_USER = os.getenv("PGUSER")
DB_PASSWORD = os.getenv("PGPASSWORD")

required_settings = {
    "PGDATABASE": DB_NAME,
    "PGUSER": DB_USER,
    "PGPASSWORD": DB_PASSWORD,
}

missing_settings = [
    name for name, value in required_settings.items()
    if not value
]

if missing_settings:
    raise ValueError(
        "Missing environment variables: "
        + ", ".join(missing_settings)
        + ". Check your .env file."
    )

database_url = URL.create(
    drivername="postgresql+psycopg2",
    username=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=int(DB_PORT),
    database=DB_NAME,
)

engine = create_engine(
    database_url,
    pool_pre_ping=True,
)

try:
    with engine.connect() as db_connection:
        database_name, username, server_version = (
            db_connection.execute(
                text("""
                    SELECT
                        current_database(),
                        current_user,
                        version()
                """)
            ).fetchone()
        )

    print("Database connection successful.")
    print(f"Database: {database_name}")
    print(f"User: {username}")
    print(f"PostgreSQL: {server_version.split(',')[0]}")

except SQLAlchemyError:
    engine.dispose()
    raise RuntimeError(
        "Database connection failed. Verify PostgreSQL is running, "
        "check your .env settings, and confirm database permissions."
    ) from None


Database connection successful.
Database: ecco_procurement
User: postgres
PostgreSQL: PostgreSQL 18.4 on x86_64-windows


In [13]:
procurement_df = pd.read_sql_query(
    """
    SELECT *
    FROM fact_procurement_monthly
    LIMIT 10;
    """,
    engine,
)

procurement_df.head()

,procurement_id,purch_org_id,material_7,material_id,vendor_id,vendor_material_id,material_description,plant_id,material_group_id,month,order_value_local,order_currency,po_qty,po_uom,gr_qty,gr_uom,unit_price_local,order_value_eur,is_zero_activity,source_system,document_grain
0,POA0000001,1001,C220000,C22000000001,100112,VM00011,Pure sock - Vitrus Mondial,1001,050122,2020-01-01,0.0,VND,0.0,PAA,0.0,PAA,NaN,0.0,1,SAP_ECCO_SIM,Monthly aggregated PO/GR
1,POA0000002,1001,C220000,C22000000001,100112,VM00011,Pure sock - Vitrus Mondial,1001,050122,2020-02-01,0.0,VND,0.0,PAA,0.0,PAA,NaN,0.0,1,SAP_ECCO_SIM,Monthly aggregated PO/GR
2,POA0000003,1001,C220000,C22000000001,100112,VM00011,Pure sock - Vitrus Mondial,1001,050122,2020-04-01,0.0,VND,0.0,PAA,0.0,PAA,NaN,0.0,1,SAP_ECCO_SIM,Monthly aggregated PO/GR
3,POA0000004,1001,C220000,C22000000001,100112,VM00011,Pure sock - Vitrus Mondial,1001,050122,2020-06-01,0.0,VND,0.0,PAA,0.0,PAA,NaN,0.0,1,SAP_ECCO_SIM,Monthly aggregated PO/GR
4,POA0000005,1001,C220000,C22000000001,100112,VM00011,Pure sock - Vitrus Mondial,1001,050122,2020-07-01,0.0,VND,0.0,PAA,0.0,PAA,NaN,0.0,1,SAP_ECCO_SIM,Monthly aggregated PO/GR
